# 🧠 Tiny GPT — From Scratch
> Train a small GPT-style language model on your own custom text dataset.
> Runs on free Colab T4 GPU.

## Architecture
- Token-level tokenizer (using HuggingFace `tokenizers` library)
- Transformer: multi-head self-attention + feedforward blocks
- Positional encoding
- Training loop with loss tracking
- Text generation with temperature control

---
## Steps
1. Upload your custom `.txt` dataset
2. Train the tokenizer on it
3. Build and train the GPT model
4. Generate text!

## ✅ Step 1 — Install Dependencies

In [ ]:
!pip install tokenizers -q
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'GPU available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')

## ✅ Step 2 — Upload Your Custom Dataset

In [ ]:
from google.colab import files
import os

print('Upload your .txt file below:')
uploaded = files.upload()

# Get the uploaded filename
DATA_FILE = list(uploaded.keys())[0]
print(f'\nUsing file: {DATA_FILE}')

with open(DATA_FILE, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f'Total characters : {len(raw_text):,}')
print(f'Total words      : {len(raw_text.split()):,}')
print(f'\nFirst 300 chars:\n{raw_text[:300]}')

## ✅ Step 3 — Train BPE Tokenizer on Your Data

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Save raw text to temp file for tokenizer training
with open('train_data.txt', 'w', encoding='utf-8') as f:
    f.write(raw_text)

# Build and train BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token='[UNK]'))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=8000,              # adjust based on dataset size
    min_frequency=2,
    special_tokens=['[UNK]', '[PAD]', '[BOS]', '[EOS]']
)

tokenizer.train(['train_data.txt'], trainer)
tokenizer.save('tokenizer.json')

VOCAB_SIZE = tokenizer.get_vocab_size()
print(f'Vocabulary size: {VOCAB_SIZE:,}')

# Test tokenizer
sample = raw_text[:100]
encoded = tokenizer.encode(sample)
print(f'\nSample text  : {sample}')
print(f'Token ids    : {encoded.ids[:20]}')
print(f'Tokens       : {encoded.tokens[:20]}')

## ✅ Step 4 — Prepare Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# ── Hyperparameters ──────────────────────────────────────────
BLOCK_SIZE   = 128    # context window (tokens)
BATCH_SIZE   = 32     # samples per batch
EMBED_DIM    = 256    # embedding dimension
NUM_HEADS    = 8      # attention heads  (EMBED_DIM must be divisible)
NUM_LAYERS   = 4      # transformer blocks
FF_DIM       = 1024   # feedforward hidden dim
DROPOUT      = 0.1
LEARNING_RATE = 3e-4
EPOCHS       = 10
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')
# ─────────────────────────────────────────────────────────────

# Tokenize entire corpus
encoded_full = tokenizer.encode(raw_text)
all_ids      = encoded_full.ids
print(f'Total tokens in corpus: {len(all_ids):,}')

# Train / val split (90/10)
split        = int(0.9 * len(all_ids))
train_ids    = all_ids[:split]
val_ids      = all_ids[split:]
print(f'Train tokens: {len(train_ids):,}  |  Val tokens: {len(val_ids):,}')


class TextDataset(Dataset):
    """Sliding window dataset — input is tokens[i:i+BLOCK_SIZE],
       target is tokens[i+1:i+BLOCK_SIZE+1] (next-token prediction)."""
    def __init__(self, token_ids, block_size):
        self.data       = torch.tensor(token_ids, dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx            : idx + self.block_size]
        y = self.data[idx + 1        : idx + self.block_size + 1]
        return x, y


train_dataset = TextDataset(train_ids, BLOCK_SIZE)
val_dataset   = TextDataset(val_ids,   BLOCK_SIZE)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

## ✅ Step 5 — Build GPT Architecture from Scratch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class MultiHeadSelfAttention(nn.Module):
    """
    Multi-head causal self-attention.
    'Causal' means each token can only attend to previous tokens
    (no peeking at future tokens) — enforced via causal mask.
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, 'embed_dim must be divisible by num_heads'

        self.num_heads  = num_heads
        self.head_dim   = embed_dim // num_heads  # dimension per head
        self.scale      = math.sqrt(self.head_dim)  # scaling factor to stabilise gradients

        # Single projection for Q, K, V (fused for efficiency)
        self.qkv_proj   = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.attn_drop  = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # batch, sequence length, embed_dim

        # Project to Q, K, V and split into heads
        qkv = self.qkv_proj(x)                          # (B, T, 3*C)
        q, k, v = qkv.split(C, dim=2)                   # each (B, T, C)

        # Reshape to (B, num_heads, T, head_dim)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention scores
        attn = (q @ k.transpose(-2, -1)) / self.scale   # (B, H, T, T)

        # Causal mask — upper triangle = -inf so softmax → 0
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn = attn.masked_fill(causal_mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        # Weighted sum of values
        out = attn @ v                                   # (B, H, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # re-merge heads
        return self.resid_drop(self.out_proj(out))


class FeedForward(nn.Module):
    """
    Position-wise feedforward network.
    Two linear layers with GELU activation in between.
    Expands to ff_dim then projects back to embed_dim.
    """
    def __init__(self, embed_dim, ff_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """
    One GPT transformer block:
      LayerNorm → MultiHeadSelfAttention → residual
      LayerNorm → FeedForward            → residual
    Pre-LayerNorm (before attention) for training stability.
    """
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.ln2  = nn.LayerNorm(embed_dim)
        self.ff   = FeedForward(embed_dim, ff_dim, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # residual connection
        x = x + self.ff(self.ln2(x))    # residual connection
        return x


class TinyGPT(nn.Module):
    """
    Full GPT model:
      Token embedding + Positional embedding
      → N x TransformerBlock
      → LayerNorm
      → Linear head (vocab_size)
    """
    def __init__(self, vocab_size, embed_dim, num_heads,
                 num_layers, ff_dim, block_size, dropout=0.1):
        super().__init__()
        self.block_size    = block_size

        self.token_emb     = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb       = nn.Embedding(block_size, embed_dim)  # learned positional
        self.drop          = nn.Dropout(dropout)

        self.blocks        = nn.Sequential(
            *[TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
              for _ in range(num_layers)]
        )
        self.ln_final      = nn.LayerNorm(embed_dim)
        self.lm_head       = nn.Linear(embed_dim, vocab_size, bias=False)

        # Weight tying — share token embedding and lm_head weights
        # Reduces parameters and improves performance
        self.lm_head.weight = self.token_emb.weight

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size

        # Token + positional embeddings
        positions  = torch.arange(T, device=idx.device).unsqueeze(0)  # (1, T)
        x          = self.drop(self.token_emb(idx) + self.pos_emb(positions))

        # Transformer blocks
        x          = self.blocks(x)
        x          = self.ln_final(x)
        logits     = self.lm_head(x)   # (B, T, vocab_size)

        # Compute loss if targets provided
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),  # (B*T, vocab_size)
                targets.view(-1)                   # (B*T,)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=1.0, top_k=50):
        """
        Autoregressively generate tokens.
        temperature: >1 = more random, <1 = more focused
        top_k: sample only from top k most likely tokens
        """
        self.eval()
        for _ in range(max_new_tokens):
            # Crop context to block_size
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)

            # Take logits at last time step
            logits = logits[:, -1, :] / temperature

            # Top-k filtering
            if top_k is not None:
                values, _ = torch.topk(logits, top_k)
                logits[logits < values[:, -1:]] = float('-inf')

            probs     = F.softmax(logits, dim=-1)
            idx_next  = torch.multinomial(probs, num_samples=1)
            idx       = torch.cat([idx, idx_next], dim=1)
        return idx


# Instantiate model
model = TinyGPT(
    vocab_size  = VOCAB_SIZE,
    embed_dim   = EMBED_DIM,
    num_heads   = NUM_HEADS,
    num_layers  = NUM_LAYERS,
    ff_dim      = FF_DIM,
    block_size  = BLOCK_SIZE,
    dropout     = DROPOUT
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')
print(model)

## ✅ Step 6 — Training Loop

In [ ]:
import time

optimizer   = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_losses = []
val_losses   = []


def evaluate(model, loader, max_batches=50):
    """Compute average validation loss over max_batches batches."""
    model.eval()
    total_loss = 0.0
    count      = 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= max_batches:
                break
            x, y        = x.to(DEVICE), y.to(DEVICE)
            _, loss     = model(x, y)
            total_loss += loss.item()
            count      += 1
    return total_loss / count


print('Starting training...\n')
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    start      = time.time()

    for step, (x, y) in enumerate(train_loader):
        x, y     = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        _, loss  = model(x, y)
        loss.backward()

        # Gradient clipping — prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        epoch_loss += loss.item()

        if step % 100 == 0:
            print(f'  Epoch {epoch} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

    scheduler.step()

    avg_train = epoch_loss / len(train_loader)
    avg_val   = evaluate(model, val_loader)
    elapsed   = time.time() - start

    train_losses.append(avg_train)
    val_losses.append(avg_val)

    print(f'\nEpoch {epoch}/{EPOCHS} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | Time: {elapsed:.1f}s\n')

print('Training complete!')

## ✅ Step 7 — Plot Loss Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses,   label='Val Loss',   marker='s')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('Tiny GPT — Training Progress')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print('Loss curve saved as loss_curve.png')

## ✅ Step 8 — Generate Text

In [ ]:
def generate_text(prompt, max_new_tokens=100, temperature=0.8, top_k=50):
    """
    Generate text from a prompt string.
    temperature: lower = more deterministic, higher = more creative
    top_k      : restrict sampling to top k tokens
    """
    model.eval()
    encoded  = tokenizer.encode(prompt)
    idx      = torch.tensor([encoded.ids], dtype=torch.long).to(DEVICE)

    output_ids = model.generate(
        idx,
        max_new_tokens = max_new_tokens,
        temperature    = temperature,
        top_k          = top_k
    )

    # Decode — only the new tokens (skip the prompt)
    new_ids  = output_ids[0].tolist()
    decoded  = tokenizer.decode(new_ids)
    return decoded


# ── Try it out ───────────────────────────────────────────────
PROMPT = "The model learned"   # ← change this to anything from your dataset

print('=' * 60)
print(f'PROMPT: {PROMPT}')
print('=' * 60)

for temp in [0.5, 0.8, 1.2]:
    print(f'\n[Temperature = {temp}]')
    result = generate_text(PROMPT, max_new_tokens=80, temperature=temp, top_k=50)
    print(result)
    print('-' * 60)

## ✅ Step 9 — Save Model Checkpoint

In [ ]:
checkpoint = {
    'model_state_dict' : model.state_dict(),
    'optimizer_state'  : optimizer.state_dict(),
    'train_losses'     : train_losses,
    'val_losses'       : val_losses,
    'hyperparams': {
        'vocab_size'  : VOCAB_SIZE,
        'embed_dim'   : EMBED_DIM,
        'num_heads'   : NUM_HEADS,
        'num_layers'  : NUM_LAYERS,
        'ff_dim'      : FF_DIM,
        'block_size'  : BLOCK_SIZE,
        'dropout'     : DROPOUT,
    }
}

torch.save(checkpoint, 'tiny_gpt_checkpoint.pt')
print('Model saved as tiny_gpt_checkpoint.pt')

# Download to local machine
from google.colab import files
files.download('tiny_gpt_checkpoint.pt')
files.download('loss_curve.png')
files.download('tokenizer.json')

## 📌 Interview Talking Points

| Component | What to say |
|---|---|
| **MultiHeadSelfAttention** | Scaled dot-product attention, causal mask prevents future token leakage, heads split embed_dim for parallel attention patterns |
| **Causal mask** | Upper-triangular matrix of -inf → after softmax those positions become 0, so token i cannot attend to token j>i |
| **Weight tying** | lm_head shares weights with token_emb — reduces parameters and improves generalisation |
| **BPE tokenizer** | Byte-Pair Encoding — iteratively merges frequent adjacent token pairs, balances vocab size vs sequence length |
| **Gradient clipping** | clip_grad_norm_(max=1.0) — prevents exploding gradients in deep transformers |
| **Temperature** | Divides logits before softmax — low temp = peaked distribution (safe), high temp = flat distribution (creative) |
| **Top-k sampling** | Only sample from k most probable tokens — avoids low-probability garbage tokens in generation |